# Sorting Notebook

This notebook will download and sort electrophysiology collected using an Intan headstage, in the .rhd format. 

The data is intracranial mouse recording, from a 16 channel microarray. The paper can be found here: https://doi.org/10.1371/journal.pone.0221510


# Getting Set Up

1. Open a terminal. Make sure "Sorter" environment is active by running these commands in the terminal 

```
conda deactivate
conda activate sorter
```


In [1]:
import sys
import torch

from pathlib import Path
import spikeinterface as si

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: /home/dhn/Documents/development/zlab_sorting_script/.venv/bin/python
PyTorch: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA T400 4GB


In [2]:
from sorting_scripts import zsort, auto_curate

In [3]:
patient = "DB007"
session = "Task_261019_165842"

path_dict = zsort.set_paths(
    "/mnt/MED_Data_iSSD/Neural_Data/DB007/Task_261019_165842"
)

rec = zsort.load_recording(path_dict)

rec = zsort.set_probe(
    rec, path_dict,
    "deeparray128_P1925-127_H1925-116.json",
)

[zsort] Loaded 15 RHD file(s): 128 channels, 30000 Hz, 25435904 samples, 847.863 seconds.
no probe, attaching one


# Load sorter and analyzer if they exist

In [ ]:
# # Paths
# SORTER_PATH = Path("/path/to/sorter_output")
# ANALYZER_PATH = Path("/path/to/sorting_analyzer")

# # Load sorting
# sorting = si.sorters.read_sorter_folder(SORTER_PATH)

# # Load analyzer
# analyzer = si.load_sorting_analyzer(ANALYZER_PATH)

# If they do not exist, run the sorter and create analyzer

In [ ]:
from pathlib import Path

retry_paths = dict(path_dict)
for key in ("sorter_output_folder", "analyzer_folder"):
    old = Path(path_dict[key])
    retry_paths[key] = old.with_name(old.name + "_lowmem")

sort = zsort.sort(
    rec, retry_paths,
    batch_size=15000,
    clear_cache=True,
)
analyzer = zsort.analyze(rec, sort, retry_paths)

[zsort] recording dtype is unsigned (uint16); converting to signed


write_binary_recording (no parallelization):   0%|          | 0/848 [00:00<?, ?it/s]

In [ ]:
# 6. Sort
sort = zsort.sort(rec, path_dict,
                  batch_size=15000,
                  clear_cache=True)



[zsort] recording dtype is unsigned (uint16); converting to signed


write_binary_recording (no parallelization):   0%|          | 0/848 [00:00<?, ?it/s]

kilosort.run_kilosort:  
kilosort.run_kilosort: Computing preprocessing variables.
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: N samples: 25435904
kilosort.run_kilosort: N seconds: 847.8634666666667
kilosort.run_kilosort: N batches: 1696
kilosort.run_kilosort: Preprocessing filters computed in 1.64s; total 1.64s
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage after preprocessing
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:     4.10 %
kilosort.run_kilosort: Mem used:     20.10 %     |      12.56 GB
kilosort.run_kilosort: Mem avail:    49.94 / 62.50 GB
kilosort.run_kilosort: ------------------------------------------------------
kilosort.run_kilosort: GPU usage:    `conda install pynvml` for GPU usage
kilosort.run_kilosort: GPU memory:   30.49 %     |      1.10   /     3.62 GB
kilosort.run_kilosort: Allocated:     0.22 %     |      0.01   /     3.62 GB
kilosort.r

kilosort4 run time 5106.60s
[zsort] recording dtype is unsigned (uint16); converting to signed


estimate_sparsity (workers: 32 processes):   0%|          | 0/848 [00:00<?, ?it/s]

...

noise_level (workers: 20 processes):   0%|          | 0/20 [00:00<?, ?it/s]

/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/postprocessing/localization_tools.py:341: RuntimeWarning: invalid value encountered in divide
  com = np.sum(wf_data[:, np.newaxis] * local_contact_locations, axis=0) / np.sum(wf_data)
/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/postprocessing/localization_tools.py:367: UserWarning: scipy.optimize.least_squares error: Each lower bound must be strictly less than each upper bound.
  warnings.warn(f"scipy.optimize.least_squares error: {e}")


Fitting PCA:   0%|          | 0/320 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/320 [00:00<?, ?it/s]

Compute : spike_amplitudes (workers: 32 processes):   0%|          | 0/848 [00:00<?, ?it/s]

In [ ]:
# 7. Analyze
analyzer = zsort.analyze(rec, sort, path_dict)

## Automated curation and ML review labels

Cleanup and stock hard merges run first. UnitRefine then labels the final unit IDs:
noise/neural for all units, followed by SUA/MUA for neural predictions only.
Predictions are advisory: the configuration below keeps all post-cleanup units.
Threshold quality labels are still computed, but neither they nor ML predictions
remove units unless their separate filtering switches are enabled.

The models were trained on mouse Neuropixels, **not validated for human DeepArray**.
`model_trust=True` allows loading the requested HF model artifacts; only use trusted
repositories. Missing cached models download automatically. Load/inference errors
stop the run and leave a diagnostic report. The current models need scikit-learn 1.4.2,
but this environment has 1.9.0: use a separate compatible environment or temporarily
set `enable_model_labeling=False`. No installed packages are changed automatically.
Training metric parameters are absent
from the current models; known settings are enforced, unknown settings are recorded.

All orchestration lives in `src`; this notebook contains only policy and review calls.
See [the workflow guide](../docs/auto_curation.md) and
[the implementation audit](../docs/model_labeling_audit.md).


In [ ]:
curation_config = auto_curate.AutoCurationConfig(
    label_method="threshold",
    thresholds={
        "snr": {"greater": 5.0},
        "firing_rate": {"greater": 0.1},
        "isi_violations_ratio": {"less": 0.5},
    },
    apply_quality_filtering=False,
    enable_model_labeling=True,
    enable_noise_classifier=True,
    enable_sua_classifier=True,
    models={
        "noise": "SpikeInterface/UnitRefine_noise_neural_classifier",
        "sua_mua": "SpikeInterface/UnitRefine_sua_mua_classifier",
    },
    model_trust=True,
    enforce_metric_params=True,
    apply_model_filtering=False,
    model_filtering={},
    remove_duplicate_spikes=True,
    remove_redundant_units=True,
    auto_merge=True,
    merge_preset="similarity_correlograms",
    export_report=True,
    export_phy=False,
    job_kwargs={"n_jobs": 1, "chunk_duration": "1s", "progress_bar": True},
)

In [ ]:
curation_result = auto_curate.auto_curate(analyzer, path_dict, curation_config)
curated_data = curation_result.analyzer
curation_result.summary()

In [ ]:
display(curation_result.ml_classifications)
display(curation_result.decisions)
display(curation_result.labels)
print("Saved analyzer:", curation_result.analyzer_folder)
print("ML report and model metadata:", curation_result.output_folder)

### Visual review

The Unit List displays separate ML labels, numeric confidence, and evaluation status.
Noise-predicted units have no fabricated SUA/MUA label. ML fields never replace human
curation labels. The companion `unit_ml_classifications.csv` uses the same final IDs
as the GUI; `decisions.csv` maps them back to source units. Each run saves independently.
Opening this viewer is read-only; no GUI Save action is needed for automated output.


In [ ]:
window = zsort.launch_curation_gui(
    curated_data, curation=False, **curation_result.gui_kwargs()
)

In [8]:
zsort.generate_figures(curated_data, path_dict)

In [ ]:
# Interact with box via api

!box folders:items 352606395707

----- Folder 352606396623 -----
Type: folder
ID: '352606396623'
Sequence ID: '0'
ETag: '0'
Name: Intan_RDH_2000

----- Folder 354522525287 -----
Type: folder
ID: '354522525287'
Sequence ID: '0'
ETag: '0'
Name: Intan_RDH_2000 (1)


In [28]:
!box folders:items 352606396623

----- Folder 352605477299 -----
Type: folder
ID: '352605477299'
Sequence ID: '0'
ETag: '0'
Name: Session1

----- Folder 352604968054 -----
Type: folder
ID: '352604968054'
Sequence ID: '0'
ETag: '0'
Name: Session2


In [29]:
!box folders:items 352605477299

----- Folder 352607353389 -----
Type: folder
ID: '352607353389'
Sequence ID: '0'
ETag: '0'
Name: raw

----- Folder 354623627238 -----
Type: folder
ID: '354623627238'
Sequence ID: '0'
ETag: '0'
Name: sorted


In [30]:
!box folders:upload 354623627238

Could not read directory 354623627238


In [58]:
!box folders:upload "/home/marco/codespace/data/Intan_RDH_2000/Session1/sorted/cleaned_analyzer.zarr" -p 354623627238

Type: folder
ID: '355655922049'
Sequence ID: '0'
ETag: '0'
Name: cleaned_analyzer.zarr
Created At: '2025-12-12T11:40:22-08:00'
Modified At: '2025-12-12T11:40:22-08:00'
Description: ''
Size: 0
Path Collection:
    Total Count: 5
    Entries:
        -
            Type: folder
            ID: '0'
            Sequence ID: null
            ETag: null
            Name: All Files
        -
            Type: folder
            ID: '352606395707'
            Sequence ID: '0'
            ETag: '0'
            Name: Cloud_Sorter
        -
            Type: folder
            ID: '352606396623'
            Sequence ID: '0'
            ETag: '0'
            Name: Intan_RDH_2000
        -
            Type: folder
            ID: '352605477299'
            Sequence ID: '0'
            ETag: '0'
            Name: Session1
        -
            Type: folder
            ID: '354623627238'
            Sequence ID: '0'
            ETag: '0'
            Name: sorted
Created By:
    Type: user
    ID: '445